# ELT - NYC Yellow Taxi (Janeiro/2024)

Esqueleto do pipeline. Estrutura sugerida:

1. Setup e conexão
2. Carga da camada raw (dados brutos, sem transformação)
3. Verificação básica da carga
4. Perfilamento dos dados 
5. Camada de tratamento / staging 
6. Modelo dimensional 
7. Camada de consumo analítico 

## 1. Setup e conexão

In [1]:
import duckdb
import os

# Caminhos dos arquivos de origem — ajustem se estiverem em outra pasta
PARQUET_PATH = "../data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_PATH = "../data/taxi_zone_lookup.csv"

# Arquivo de banco persistente em disco (fica salvo entre execuções)
DB_PATH = "nyc_taxi.duckdb"

con = duckdb.connect(DB_PATH)

## 2. Carga da camada raw (dados brutos, sem transformação)

A camada raw deve ser uma cópia fiel do
que chegou, para fins de auditoria e rastreabilidade. 

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_yellow_tripdata AS
    SELECT *
    FROM read_parquet('{PARQUET_PATH}')
""")

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_taxi_zone_lookup AS
    SELECT *
    FROM read_csv_auto('{ZONE_LOOKUP_PATH}')
""")

## 3. Verificação básica da carga


In [4]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_yellow_tripdata").fetchdf()

,total_linhas
0,2964624


In [5]:
con.execute("SELECT * FROM raw_yellow_tripdata LIMIT 5").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [6]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_taxi_zone_lookup").fetchdf()

,total_linhas
0,265


In [7]:
con.execute("SELECT * FROM raw_taxi_zone_lookup LIMIT 5").fetchdf()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [8]:
con.execute("DESCRIBE raw_yellow_tripdata").fetchdf()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [9]:
con.execute("DESCRIBE raw_taxi_zone_lookup").fetchdf()

,column_name,column_type,null,key,default,extra
0,LocationID,BIGINT,YES,None,None,None
1,Borough,VARCHAR,YES,None,None,None
2,Zone,VARCHAR,YES,None,None,None
3,service_zone,VARCHAR,YES,None,None,None


## 4. Perfilamento dos dados

Coisas a investigar em `raw_yellow_tripdata` e `raw_taxi_zone_lookup`:
- Nulos por coluna
- Duplicidade de registros
- Cardinalidade e domínio das colunas categóricas (ex.: payment_type, RatecodeID)
- Valores improváveis (datas fora de janeiro/2024, distâncias/valores negativos, etc.)
- Correspondência dos códigos de localização entre os dois arquivos

Guardem aqui as consultas usadas e o que cada uma revelou.

# TODO: Perfilamento

### 4.1 Valores Nulos

In [10]:
con.execute("""
    SELECT COUNT(*) AS total_linhas,
           COUNT(*) - COUNT(VendorID) AS nulos_vendorid,
           COUNT(*) - COUNT(tpep_pickup_datetime) AS nulos_pickup_datetime,
           COUNT(*) - COUNT(tpep_dropoff_datetime) AS nulos_dropoff_datetime,
           COUNT(*) - COUNT(passenger_count) AS nulos_passenger_count,
           COUNT(*) - COUNT(trip_distance) AS nulos_trip_distance,
           COUNT(*) - COUNT(RatecodeID) AS nulos_ratecodeid,
           COUNT(*) - COUNT(store_and_fwd_flag) AS nulos_store_and_fwd_flag,
           COUNT(*) - COUNT(PULocationID) AS nulos_pulocationid,
           COUNT(*) - COUNT(DOLocationID) AS nulos_dolocationid,
           COUNT(*) - COUNT(payment_type) AS nulos_payment_type,
           COUNT(*) - COUNT(fare_amount) AS nulos_fare_amount,
           COUNT(*) - COUNT(extra) AS nulos_extra,
           COUNT(*) - COUNT(mta_tax) AS nulos_mta_tax,
           COUNT(*) - COUNT(tip_amount) AS nulos_tip_amount,
           COUNT(*) - COUNT(tolls_amount) AS nulos_tolls_amount,
           COUNT(*) - COUNT(improvement_surcharge) AS nulos_improvement_surcharge,
           COUNT(*) - COUNT(total_amount) AS nulos_total_amount,
           COUNT(*) - COUNT(congestion_surcharge) AS nulos_congestion_surcharge,
           COUNT(*) - COUNT(Airport_fee) AS nulos_airport_fee
    FROM raw_yellow_tripdata
""").fetchdf()

,total_linhas,nulos_vendorid,nulos_pickup_datetime,nulos_dropoff_datetime,nulos_passenger_count,nulos_trip_distance,nulos_ratecodeid,nulos_store_and_fwd_flag,nulos_pulocationid,nulos_dolocationid,nulos_payment_type,nulos_fare_amount,nulos_extra,nulos_mta_tax,nulos_tip_amount,nulos_tolls_amount,nulos_improvement_surcharge,nulos_total_amount,nulos_congestion_surcharge,nulos_airport_fee
0,2964624,0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


In [26]:
con.execute("""
    SELECT
        COUNT(*) AS linhas_com_todas_nulas
    FROM raw_yellow_tripdata
    WHERE passenger_count IS NULL
      AND RatecodeID IS NULL
      AND store_and_fwd_flag IS NULL
      AND congestion_surcharge IS NULL
      AND Airport_fee IS NULL
""").fetchdf()

,linhas_com_todas_nulas
0,140162


### 4.2 Linhas Duplicadas

In [11]:
con.execute("""
    SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
           PULocationID, DOLocationID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID
    HAVING COUNT(*) > 1
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,qtd
0,2,2024-01-13 07:33:00,2024-01-13 07:35:03,236,75,2
1,2,2024-01-13 08:37:46,2024-01-13 08:54:58,249,97,2
2,2,2024-01-13 10:56:42,2024-01-13 11:13:36,237,211,2
3,2,2024-01-13 11:41:26,2024-01-13 11:45:56,142,151,2
4,2,2024-01-13 12:24:23,2024-01-13 12:38:36,48,113,2
...,...,...,...,...,...,...
31104,2,2024-01-29 12:28:53,2024-01-29 12:48:40,141,42,2
31105,2,2024-01-29 12:17:39,2024-01-29 12:28:35,239,163,2
31106,2,2024-01-29 13:58:33,2024-01-29 14:07:43,237,140,2
31107,2,2024-01-29 14:04:50,2024-01-29 14:08:21,41,41,2


### 4.3 Payment_type

In [12]:
con.execute("""
    SELECT payment_type, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY payment_type
    ORDER BY qtd DESC
""").fetchdf()

,payment_type,qtd
0,1,2319046
1,2,439191
2,0,140162
3,4,46628
4,3,19597


### 4.4 RatecodeID

In [13]:
con.execute("""
    SELECT RatecodeID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY RatecodeID
    ORDER BY qtd DESC
""").fetchdf()

,RatecodeID,qtd
0,1,2663350
1,<NA>,140162
2,2,98713
3,99,28663
4,5,19410
5,3,7954
6,4,6365
7,6,7


### 4.5 VendorID

In [14]:
con.execute("""
    SELECT VendorID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,qtd
0,2,2234632
1,1,729732
2,6,260


### 4.6 Passenger_count

In [15]:
con.execute("""
    SELECT passenger_count, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY passenger_count
    ORDER BY passenger_count
""").fetchdf()

,passenger_count,qtd
0,0,31465
1,1,2188739
2,2,405103
3,3,91262
4,4,51974
5,5,33506
6,6,22353
7,7,8
8,8,51
9,9,1


### 4.7 store_and_fwd_flag

In [16]:
con.execute("""
    SELECT store_and_fwd_flag, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY store_and_fwd_flag
    ORDER BY qtd DESC
""").fetchdf()

,store_and_fwd_flag,qtd
0,N,2813126
1,NaN,140162
2,Y,11336


---

### 4.8 Valores max/min

In [17]:
con.execute("""
    SELECT
        MIN(trip_distance) AS min_distancia, MAX(trip_distance) AS max_distancia,
        MIN(fare_amount) AS min_tarifa, MAX(fare_amount) AS max_tarifa,
        MIN(extra) AS min_extra, MAX(extra) AS max_extra,
        MIN(mta_tax) AS min_mta_tax, MAX(mta_tax) AS max_mta_tax,
        MIN(tip_amount) AS min_tip, MAX(tip_amount) AS max_tip,
        MIN(tolls_amount) AS min_tolls, MAX(tolls_amount) AS max_tolls,
        MIN(improvement_surcharge) AS min_improvement, MAX(improvement_surcharge) AS max_improvement,
        MIN(total_amount) AS min_total, MAX(total_amount) AS max_total,
        MIN(congestion_surcharge) AS min_congestion, MAX(congestion_surcharge) AS max_congestion,
        MIN(Airport_fee) AS min_airport_fee, MAX(Airport_fee) AS max_airport_fee
    FROM raw_yellow_tripdata
""").fetchdf().T

,0
min_distancia,0.00
max_distancia,312722.30
min_tarifa,-899.00
max_tarifa,5000.00
min_extra,-7.50
max_extra,14.25
min_mta_tax,-0.50
max_mta_tax,4.00
min_tip,-80.00
max_tip,428.00


### 4.8 Média e Mediana

In [18]:
con.execute("""
    SELECT
        ROUND(AVG(trip_distance), 2) AS media_distancia, MEDIAN(trip_distance) AS mediana_distancia,
        ROUND(AVG(fare_amount), 2) AS media_tarifa, MEDIAN(fare_amount) AS mediana_tarifa,
        ROUND(AVG(extra), 2) AS media_extra, MEDIAN(extra) AS mediana_extra,
        ROUND(AVG(mta_tax), 2) AS media_mta_tax, MEDIAN(mta_tax) AS mediana_mta_tax,
        ROUND(AVG(tip_amount), 2) AS media_tip, MEDIAN(tip_amount) AS mediana_tip,
        ROUND(AVG(tolls_amount), 2) AS media_tolls, MEDIAN(tolls_amount) AS mediana_tolls,
        ROUND(AVG(improvement_surcharge), 2) AS media_improvement, MEDIAN(improvement_surcharge) AS mediana_improvement,
        ROUND(AVG(total_amount), 2) AS media_total, MEDIAN(total_amount) AS mediana_total,
        ROUND(AVG(congestion_surcharge), 2) AS media_congestion, MEDIAN(congestion_surcharge) AS mediana_congestion,
        ROUND(AVG(Airport_fee), 2) AS media_airport_fee, MEDIAN(Airport_fee) AS mediana_airport_fee,
        ROUND(AVG(passenger_count), 2) AS media_passageiros
    FROM raw_yellow_tripdata
""").fetchdf().T

,0
media_distancia,3.65
mediana_distancia,1.68
media_tarifa,18.18
mediana_tarifa,12.80
media_extra,1.45
mediana_extra,1.00
media_mta_tax,0.48
mediana_mta_tax,0.50
media_tip,3.34
mediana_tip,2.70


---

### 4.9 Desembarque antes do Embarque

In [19]:
con.execute("""
    SELECT COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
""").fetchdf()

,qtd
0,56


In [20]:
con.execute("""
    SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
           trip_distance, fare_amount, PULocationID, DOLocationID
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
    ORDER BY tpep_pickup_datetime
    LIMIT 10
""").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,PULocationID,DOLocationID
0,6,2024-01-01 02:01:50,2024-01-01 02:01:39,7.46,39.22,265,197
1,6,2024-01-01 03:01:46,2024-01-01 03:01:39,7.65,40.69,265,62
2,1,2024-01-02 10:00:00,2024-01-02 09:53:56,7.20,38.50,75,95
3,6,2024-01-02 16:01:30,2024-01-02 16:01:19,9.02,49.82,265,95
4,6,2024-01-02 16:01:44,2024-01-02 16:01:24,7.32,44.20,265,129
5,6,2024-01-03 05:01:56,2024-01-03 05:01:27,4.63,18.43,265,130
6,6,2024-01-03 16:01:34,2024-01-03 16:01:21,3.90,31.20,265,260
7,6,2024-01-03 18:01:49,2024-01-03 18:01:44,8.09,39.20,265,168
8,1,2024-01-04 12:30:00,2024-01-04 12:18:23,4.80,31.50,165,222
9,1,2024-01-04 14:00:00,2024-01-04 13:55:50,5.60,27.50,95,80


In [21]:
con.execute("""
    SELECT VendorID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
    GROUP BY VendorID
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,qtd
0,6,48
1,1,8


---

### 4.10 Outliers

In [25]:
con.execute("""
    SELECT
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(trip_distance, 0.25) AS q1,
                        quantile_cont(trip_distance, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                trip_distance < l.q1 - 1.5 * (l.q3 - l.q1)
                OR trip_distance > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_trip_distance,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(fare_amount, 0.25) AS q1,
                        quantile_cont(fare_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                fare_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR fare_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_fare_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(extra, 0.25) AS q1,
                        quantile_cont(extra, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                extra < l.q1 - 1.5 * (l.q3 - l.q1)
                OR extra > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_extra,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(mta_tax, 0.25) AS q1,
                        quantile_cont(mta_tax, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                mta_tax < l.q1 - 1.5 * (l.q3 - l.q1)
                OR mta_tax > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_mta_tax,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(tip_amount, 0.25) AS q1,
                        quantile_cont(tip_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                tip_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR tip_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_tip_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(tolls_amount, 0.25) AS q1,
                        quantile_cont(tolls_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                tolls_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR tolls_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_tolls_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(improvement_surcharge, 0.25) AS q1,
                        quantile_cont(improvement_surcharge, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                improvement_surcharge < l.q1 - 1.5 * (l.q3 - l.q1)
                OR improvement_surcharge > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_improvement_surcharge,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(total_amount, 0.25) AS q1,
                        quantile_cont(total_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                total_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR total_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_total_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(congestion_surcharge, 0.25) AS q1,
                        quantile_cont(congestion_surcharge, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                congestion_surcharge < l.q1 - 1.5 * (l.q3 - l.q1)
                OR congestion_surcharge > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_congestion_surcharge,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(Airport_fee, 0.25) AS q1,
                        quantile_cont(Airport_fee, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                Airport_fee < l.q1 - 1.5 * (l.q3 - l.q1)
                OR Airport_fee > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_airport_fee,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(passenger_count, 0.25) AS q1,
                        quantile_cont(passenger_count, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                passenger_count < l.q1 - 1.5 * (l.q3 - l.q1)
                OR passenger_count > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_passenger_count
""").fetchdf().T

,0
outliers_trip_distance,382745
outliers_fare_amount,318801
outliers_extra,47566
outliers_mta_tax,64150
outliers_tip_amount,216756
outliers_tolls_amount,210815
outliers_improvement_surcharge,36914
outliers_total_amount,363621
outliers_congestion_surcharge,246707
outliers_airport_fee,237673


## 5. Camada de tratamento (staging)

*( cada transformação deve ter justificativa
baseada no que foi encontrado no perfilamento acima.)*

In [22]:
# TODO: tratamento / staging

## 6. Modelo dimensional

*( declarem aqui a granularidade da fato antes
de criar qualquer tabela.)*

- Grão da fato:
- Dimensões planejadas:

In [23]:
# TODO: tabela fato e dimensões

## 7. Camada de consumo analítico

*(consultas de negócio sobre tempo, geografia,
financeiro e operação, sempre a partir do modelo dimensional.)*

In [24]:
# TODO: consultas analíticas